In [92]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field


In [93]:
load_dotenv()

True

In [94]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [95]:
class SentimentSchema(BaseModel):
    sentiment:  Literal["positive", "negative"] = Field(description= 'sentiment of the review')

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description="The category of issue mentioned in the review")
    tone: Literal['angry', 'frustrated', 'disappointed', 'calm']= Field(description= "The emotional tone expressed by the user")
    urgency: Literal['low', 'medium', 'high']= Field(description= "How urgent or critical the  issue appears to be.")

In [96]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [97]:
# prompt = 'what is the sentiment of the following review - The software is barely ok'
# structured_model.invoke(prompt).sentiment

In [98]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: str
    response: str

In [99]:
def find_sentiment(state: ReviewState):
    prompt= f'For the following review find the following sentiment - {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment' : sentiment}

def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state: ReviewState):
    prompt = f""" Write a warm thank you message in response to this review: {state['review']}..
    Also, kindly ask user to leave feedback on our website."""

    response = model.invoke(prompt)

    return {'response' : response.content}

def run_diagnosis(state: ReviewState):
    
    prompt = f"""Diagnose this negative review: {state['review']} \n
    return issue_type, tone, and urgency"""

    response = structured_model2.invoke(prompt)

    return {'diagnosis' : response.model_dump()}

def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f""" The user had a {diagnosis['issue_type']} issue, sounded {diagnosis['tone']}, and marked 
    urgency as {diagnosis['urgency']}. Write empathetic and helpful resolution message."""

    response = model.invoke(prompt).content
    return {'response' : response}


In [100]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)



graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response',END)



workflow = graph.compile()

In [102]:
initial_state = {
    'review': """I’m really disappointed with this product. From the moment I started using it, it felt cheap and poorly made. It didn’t work as advertised and caused more frustration than convenience. For the price I paid, 
    I expected much better quality and performance. Overall, it was a waste of money and I definitely wouldn’t recommend it to anyone."""
}

workflow.invoke(initial_state)

{'review': 'I’m really disappointed with this product. From the moment I started using it, it felt cheap and poorly made. It didn’t work as advertised and caused more frustration than convenience. For the price I paid, \n    I expected much better quality and performance. Overall, it was a waste of money and I definitely wouldn’t recommend it to anyone.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'UX', 'tone': 'disappointed', 'urgency': 'high'},
 'response': "Oh no! I'm so sorry to hear you've run into a UX issue, and I completely understand how frustrating and disappointing that must be, especially when it's impacting your work or experience to the point of high urgency.\n\nPlease accept our sincerest apologies for this inconvenience. Experiencing a problem that disrupts your flow is absolutely not what we want for you, and we take your feedback, especially when it's urgent, very seriously.\n\nI've immediately flagged this with our [relevant team, e.g., engineering/produc